In [3]:
# CLASSICAL DECODING: Recover secrets from shares
# Implements equations (5) and (6) from paper

import cv2
import numpy as np
import os

print(f"\n{'='*60}")
print("DECODING PHASE - Recovering Secrets")
print("="*60)

# Load universal share
if not os.path.exists('universal_share.png'):
    print("❌ Error: universal_share.png not found. Run encoding first!")
    exit(1)

universal_share = cv2.imread('universal_share.png', cv2.IMREAD_GRAYSCALE)
TARGET_SIZE = universal_share.shape
rows, cols = TARGET_SIZE

print(f"\n📥 Loading shares...")
print(f"   Universal share: {TARGET_SIZE}")

# Load participant shares
participant_shares = []
k = 1
while os.path.exists(f'participant_share_{k}.png'):
    share = cv2.imread(f'participant_share_{k}.png', cv2.IMREAD_GRAYSCALE)
    participant_shares.append(share)
    print(f"   Participant share {k}: {share.shape}")
    k += 1

n_secrets = len(participant_shares)
print(f"\n   Total shares loaded: {n_secrets + 1}")
print(f"   Secrets to recover: {n_secrets}")

# Paper's Decoding Equations:
# G_n[1:4] = S1[1:4]                                           (LSB 4 bits)
# G_n[5:8] = S1[5:8] ⊕ S2[5:8] ⊕ ... ⊕ Sn+1[1:4]              (MSB 4 bits)

all_shares = [universal_share] + participant_shares

print(f"\n🔓 Recovering secrets using Boolean XOR...\n")

recovered_secrets = []

for k in range(n_secrets):
    print(f"   Recovering Secret {k+1} (G{k+1}):")
    print(f"      Formula: G{k+1}[1:4] = S1[1:4]")
    print(f"               G{k+1}[5:8] = S1[5:8] ⊕ S2[5:8] ⊕ ... ⊕ S{k+2}[1:4]")
    
    recovered_secret = np.zeros(TARGET_SIZE, dtype=np.uint8)
    
    for i in range(rows):
        for j in range(cols):
            # Lower 4 bits from universal share
            g_lower = all_shares[0][i, j] & 0x0F
            
            # Upper 4 bits: XOR chain
            g_upper = (all_shares[0][i, j] & 0xF0) >> 4
            
            # XOR with upper 4 bits of shares S2 to S_{k+1}
            for idx in range(1, k+1):
                g_upper ^= (all_shares[idx][i, j] & 0xF0) >> 4
            
            # XOR with lower 4 bits of S_{k+2}
            if k+1 < len(all_shares):
                g_upper ^= (all_shares[k+1][i, j] & 0x0F)
            
            # Combine
            recovered_secret[i, j] = (g_upper << 4) | g_lower
    
    recovered_secrets.append(recovered_secret)
    
    # Save recovered secret
    cv2.imwrite(f'recovered_secret_{k+1}.png', recovered_secret)
    print(f"      ✅ Saved recovered_secret_{k+1}.png")
    print(f"         Pixel range: [{recovered_secret.min()}, {recovered_secret.max()}]")
    
    # Calculate PSNR if original exists
    original_path = f'secret_{k+1}.tif'
    if os.path.exists(original_path):
        original = cv2.imread(original_path, cv2.IMREAD_GRAYSCALE)
        original_resized = cv2.resize(original, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        
        mse = np.mean((original_resized.astype(float) - recovered_secret.astype(float)) ** 2)
        
        if mse == 0:
            print(f"         📊 PSNR: ∞ dB (PERFECT RECOVERY!)")
        else:
            psnr = 20 * np.log10(255.0 / np.sqrt(mse))
            print(f"         📊 PSNR: {psnr:.2f} dB", end="")
            
            if psnr > 35:
                print(" (Excellent - Almost Identical)")
            elif psnr > 30:
                print(" (Very Good)")
            elif psnr > 25:
                print(" (Acceptable)")
            else:
                print(" (Poor - Check implementation)")
        
        # Show difference statistics
        diff = np.abs(original_resized.astype(float) - recovered_secret.astype(float))
        print(f"         Max pixel error: {diff.max():.1f}")
        print(f"         Avg pixel error: {diff.mean():.2f}")
    else:
        print(f"         ⚠️  Original secret_{k+1}.png not found, cannot compute PSNR")
    
    print()

print("="*60)
print("✅ DECODING COMPLETE!")
print("="*60)
print(f"\nRecovered {n_secrets} secrets:")
for k in range(n_secrets):
    print(f"   - recovered_secret_{k+1}.png")
print("\nCompare recovered secrets with original secret images!")



DECODING PHASE - Recovering Secrets

📥 Loading shares...
   Universal share: (512, 512)
   Participant share 1: (512, 512)
   Participant share 2: (512, 512)

   Total shares loaded: 3
   Secrets to recover: 2

🔓 Recovering secrets using Boolean XOR...

   Recovering Secret 1 (G1):
      Formula: G1[1:4] = S1[1:4]
               G1[5:8] = S1[5:8] ⊕ S2[5:8] ⊕ ... ⊕ S2[1:4]
      ✅ Saved recovered_secret_1.png
         Pixel range: [0, 255]
         📊 PSNR: 31.59 dB (Very Good)
         Max pixel error: 15.0
         Avg pixel error: 5.47

   Recovering Secret 2 (G2):
      Formula: G2[1:4] = S1[1:4]
               G2[5:8] = S1[5:8] ⊕ S2[5:8] ⊕ ... ⊕ S3[1:4]
      ✅ Saved recovered_secret_2.png
         Pixel range: [0, 255]
         📊 PSNR: 30.83 dB (Very Good)
         Max pixel error: 15.0
         Avg pixel error: 6.10

✅ DECODING COMPLETE!

Recovered 2 secrets:
   - recovered_secret_1.png
   - recovered_secret_2.png

Compare recovered secrets with original secret images!
